# Prácticas con OpenRPS: propagación de radio y canal multitrayecto

En este cuaderno usarás **OpenRPS** para explorar cuatro ideas básicas de propagación:

1. **Práctica 1:** cómo crece la pérdida de trayecto con la distancia según el modelo (FSPL, log-distancia, Two-Ray, Okumura-Hata).
2. **Práctica 2:** mapas de cobertura con varias estaciones base y el efecto de un muro reflector.
3. **Práctica 3:** respuesta al impulso del canal (CIR): perfil de potencia-retardo, dispersión de retardo y dispersión angular.
4. **Práctica 4:** suma coherente de los rayos (desvanecimiento rápido) y mapas de SINR con antenas isotrópicas y sectoriales.

Cada práctica tiene código listo para ejecutar, **preguntas** y una sección **Tu turno** con ejercicios que debes completar modificando parámetros.

> **Instalación:** desde la carpeta del proyecto, `pip install -e ".[notebook]"`.
> Ejecuta las celdas en orden (**Kernel → Restart & Run All** si algo se ve raro).

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt

from open_rps import (
    PropagationSimulator, Transmitter, PlanarReflector,
    FreeSpacePathLoss, LogDistancePathLoss, TwoRayGroundReflection,
    OkumuraHata, Cost231Hata,
)

plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True, "grid.alpha": 0.3})

---
## Práctica 1: pérdida de trayecto vs. distancia

La **pérdida de trayecto** (*path loss*) mide cuántos dB se atenúa la señal entre transmisor y receptor.
Los modelos de OpenRPS comparten la misma interfaz: `modelo.path_loss(distancia_m, frecuencia_hz, ht_m=..., hr_m=...)`.

Comparamos varios modelos a **900 MHz**, con antena de la estación base a 30 m y móvil a 1.5 m,
entre 1 y 10 km (el rango donde Okumura-Hata es válido).

In [ ]:
f = 900e6                                  # Hz
d = np.logspace(3, 4, 200)                 # 1 km ... 10 km, en metros

modelos = {
    "Espacio libre (FSPL)":      FreeSpacePathLoss(),
    "Log-distancia (n = 3.5)":   LogDistancePathLoss(path_loss_exponent=3.5),
    "Two-Ray":                   TwoRayGroundReflection(),
    "Okumura-Hata urbano":       OkumuraHata("urban_medium"),
    "Okumura-Hata suburbano":    OkumuraHata("suburban"),
    "Okumura-Hata rural":        OkumuraHata("open"),
}

pl = {nombre: m.path_loss(d, f, ht_m=30.0, hr_m=1.5) for nombre, m in modelos.items()}

fig, ax = plt.subplots()
for nombre, valores in pl.items():
    ax.semilogx(d / 1000, valores, label=nombre)
ax.set_xlabel("Distancia [km]")
ax.set_ylabel("Pérdida de trayecto [dB]")
ax.set_title("Pérdida de trayecto a 900 MHz (ht = 30 m, hr = 1.5 m)")
ax.legend(fontsize=8)
plt.show()

### Pendiente de cada modelo

En escala logarítmica, la pérdida crece como $10\,n\log_{10}(d)$: la pendiente en **dB por década** de distancia es $10\,n$.
Estimamos $n$ ajustando una recta a cada curva:

In [ ]:
print(f"{'Modelo':28s} {'dB/década':>10s} {'n equivalente':>14s}")
for nombre, valores in pl.items():
    pendiente = np.polyfit(np.log10(d), valores, 1)[0]
    print(f"{nombre:28s} {pendiente:10.1f} {pendiente/10:14.2f}")

**Preguntas**

1. ¿Qué exponente $n$ obtienes para el espacio libre? ¿Coincide con la teoría?
2. ¿Por qué Two-Ray tiene una pendiente mucho mayor que el espacio libre? ¿Cerca de qué valor de $n$ se acerca a grandes distancias?
3. Ordena los entornos de Okumura-Hata (urbano, suburbano, rural) de mayor a menor pérdida. ¿Tiene sentido físico?

### Tu turno

- **(a)** Cambia la altura de la antena base `ht_m` de 30 m a 60 m. ¿Cuántos dB gana Okumura-Hata? ¿Y Two-Ray?
- **(b)** Repite la gráfica a **1800 MHz** usando `Cost231Hata("urban_medium")` en lugar de `OkumuraHata` (Okumura-Hata solo es válido hasta 1500 MHz). ¿Cuántos dB más de pérdida hay respecto a 900 MHz a 5 km?
- **(c)** Si a 5 km necesitas una pérdida máxima de 130 dB, ¿qué modelos la cumplen? ¿Qué significa eso para el diseño de una red?

Prueba qué pasa si evalúas `OkumuraHata` a 2.4 GHz: verás una advertencia (`ModelRangeWarning`) porque estás fuera de su rango de validez.

In [ ]:
# Espacio para tus ejercicios (a), (b) y (c)

---
## Práctica 2: cobertura con varias celdas y un muro reflector

Ahora simulamos una zona de **1 km × 1 km** a 2.4 GHz con **dos estaciones base** y, opcionalmente, un **muro reflector** vertical en $x = 500$ m.
El simulador calcula, en cada punto de la cuadrícula, los rayos directo, reflejado en el suelo y reflejado en el muro (trazado de rayos 3D).

La *celda servidora* de cada punto es la estación que entrega mayor potencia.

In [ ]:
def construir_simulador(con_muro=True, reflectividad=0.6):
    sim = PropagationSimulator()
    sim.add_transmitter(Transmitter(tx_id=0, x=200, y=250, z=25, tx_power_dbm=43, frequency_hz=2.4e9, antenna_gain_dbi=12))
    sim.add_transmitter(Transmitter(tx_id=1, x=800, y=750, z=25, tx_power_dbm=40, frequency_hz=2.4e9, antenna_gain_dbi=12))
    if con_muro:
        sim.add_reflector(PlanarReflector(p0=[500, 500, 0], normal=[1, 0, 0], reflectivity=reflectividad, name="Fachada"))
    return sim

def simular(sim, resolucion=10.0):
    return sim.run_raytracing_simulation(x_range=(0, 1000), y_range=(0, 1000), resolution_m=resolucion, rx_height_m=1.5)

res_sin = simular(construir_simulador(con_muro=False))
res_con = simular(construir_simulador(con_muro=True))

print(f"Rejilla: {len(res_con.x_coords)} x {len(res_con.y_coords)} puntos")
print(f"Potencia media sin muro: {np.nanmean(res_sin.max_rx_power_dbm):.2f} dBm")
print(f"Potencia media con muro: {np.nanmean(res_con.max_rx_power_dbm):.2f} dBm")

In [ ]:
ext = [res_con.x_coords[0], res_con.x_coords[-1], res_con.y_coords[0], res_con.y_coords[-1]]
vmin = min(np.nanmin(res_sin.max_rx_power_dbm), np.nanmin(res_con.max_rx_power_dbm))
vmax = max(np.nanmax(res_sin.max_rx_power_dbm), np.nanmax(res_con.max_rx_power_dbm))

fig, axs = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)
for ax, res, titulo in [(axs[0], res_sin, "Sin muro"), (axs[1], res_con, "Con muro")]:
    im = ax.imshow(res.max_rx_power_dbm, origin="lower", extent=ext, vmin=vmin, vmax=vmax, cmap="viridis")
    ax.set_title(f"Potencia recibida [dBm]: {titulo}")
fig.colorbar(im, ax=axs[:2], shrink=0.85, label="dBm")

dif = res_con.max_rx_power_dbm - res_sin.max_rx_power_dbm
im2 = axs[2].imshow(dif, origin="lower", extent=ext, cmap="magma")
axs[2].set_title("Diferencia con muro - sin muro [dB]")
fig.colorbar(im2, ax=axs[2], shrink=0.85, label="dB")

for ax in axs:
    ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")
    ax.axvline(500, color="w", ls="--", lw=0.8)       # posición del muro
    for tx in construir_simulador().transmitters:
        ax.plot(tx.x, tx.y, "r^", ms=9)               # estaciones base
plt.show()

### Celda servidora y dispersión de retardo

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)

axs[0].imshow(res_con.best_server_id, origin="lower", extent=ext, cmap="coolwarm")
axs[0].set_title("Celda servidora (azul = Tx0, rojo = Tx1)")

im = axs[1].imshow(res_con.rms_delay_spread_ns, origin="lower", extent=ext, cmap="plasma")
axs[1].set_title("Dispersión de retardo RMS [ns] (con muro)")
fig.colorbar(im, ax=axs[1], shrink=0.85, label="ns")
for ax in axs:
    ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")
plt.show()

**Preguntas**

1. En el mapa de diferencias, ¿en qué zonas aporta el muro más potencia y por qué? ¿Y dónde no aporta nada? (Pista: la reflexión especular solo existe si transmisor y receptor están **del mismo lado** del muro.)
2. ¿Dónde está el límite entre las dos celdas? ¿Por qué no es una recta a mitad de camino entre las dos estaciones?
3. ¿Dónde es mayor la dispersión de retardo? ¿Qué implica para un sistema de banda ancha?

> **Ojo con el tipo de suma:** por defecto OpenRPS suma las potencias de los rayos de forma **incoherente** (ignora la fase), por lo que en estos mapas no aparecen desvanecimientos por interferencia: lo que ves es la potencia *media local*, y por eso el muro solo puede *aumentarla*. En un canal real los rayos se suman con su fase y aparecen desvanecimientos rápidos; en la **Práctica 4** los verás activando `coherent=True`.

### Tu turno

- **(a)** Cambia `reflectividad` a 0.2 y a 1.0. ¿Cómo cambia la potencia media y la diferencia máxima?
- **(b)** Limita el muro a un tramo con `bounds=(xmin, xmax, ymin, ymax, zmin, zmax)`, por ejemplo `bounds=(500, 500, 300, 700, 0, 30)` (edificio de 400 m de ancho y 30 m de alto). ¿Qué cambia en el mapa de diferencias?
- **(c)** Usa el modelo de Fresnel del muro en lugar de la reflectividad fija: `PlanarReflector(..., eps_r=5.0, sigma=0.01)` (hormigón aproximado).
- **(d)** Mueve el Tx1 a $(500, 900)$. ¿Cómo cambian las celdas y la dispersión de retardo?

In [ ]:
# Espacio para tus ejercicios (a)-(d)

---
## Práctica 3: respuesta al impulso del canal (CIR)

En un punto concreto, el canal se describe como un conjunto de **rayos**, cada uno con amplitud, retardo y ángulos de salida/llegada:

$$h(t) = \sum_k a_k\, e^{j\theta_k}\, \delta(t - \tau_k)$$

De ahí salen las métricas clásicas: potencia total, **retardo medio**, **dispersión de retardo RMS** $\sigma_\tau$ y **dispersión angular**.

In [ ]:
sim = construir_simulador(con_muro=True)
tx = sim.transmitters[0]
rx = (400.0, 300.0, 1.5)               # receptor al oeste del muro, del mismo lado que Tx0

cir = sim.ray_tracer.trace_paths(tx, rx)

etiquetas = ["Directo", "Suelo"] + [f"Muro {i+1}" for i in range(len(cir.rays) - 2)]
print(f"{'Rayo':10s} {'Retardo [ns]':>13s} {'Potencia [dBm]':>15s} {'AoD azim. [°]':>14s} {'AoA azim. [°]':>14s}")
for nombre, r in zip(etiquetas, cir.rays):
    p_dbm = 10 * np.log10(r.power_linear)
    print(f"{nombre:10s} {r.tau_ns:13.2f} {p_dbm:15.2f} {r.tx_phi_deg:14.1f} {r.rx_phi_deg:14.1f}")

_, sigma_phi = cir.angular_spread(plane="phi", station="bs")
print()
print(f"Potencia total recibida:       {cir.total_received_power_dbm():.2f} dBm")
print(f"Retardo medio (absoluto):      {cir.mean_excess_delay():.2f} ns")
print(f"Retardo medio en exceso:       {cir.mean_excess_delay(relative_to_first=True):.2f} ns   (respecto al primer rayo)")
print(f"Dispersión de retardo RMS:     {cir.rms_delay_spread():.2f} ns")
print(f"Dispersión angular (BS, azim): {sigma_phi:.2f} °")

In [ ]:
tau = np.array([r.tau_ns for r in cir.rays])
p_dbm = np.array([10 * np.log10(r.power_linear) for r in cir.rays])

fig, ax = plt.subplots()
ax.stem(tau - tau.min(), p_dbm, basefmt=" ")
for t, p, nombre in zip(tau - tau.min(), p_dbm, etiquetas):
    ax.annotate(nombre, (t, p), textcoords="offset points", xytext=(5, 5))
ax.set_xlabel("Retardo en exceso [ns]")
ax.set_ylabel("Potencia del rayo [dBm]")
ax.set_title("Perfil de potencia-retardo (PDP)")
plt.show()

### Recorrido a lo largo de una línea

Movemos el receptor de oeste a este (y = 300 m) y calculamos la dispersión de retardo en cada punto, con y sin muro.

In [ ]:
xs = np.arange(0, 1001, 5.0)

def ds_a_lo_largo(sim, tx_idx=0, y=300.0):
    tx = sim.transmitters[tx_idx]
    return np.array([sim.ray_tracer.trace_paths(tx, (x, y, 1.5)).rms_delay_spread() for x in xs])

ds_con = ds_a_lo_largo(construir_simulador(True))
ds_sin = ds_a_lo_largo(construir_simulador(False))

fig, ax = plt.subplots()
ax.plot(xs, ds_sin, label="Sin muro")
ax.plot(xs, ds_con, label="Con muro (x = 500 m)")
ax.axvline(500, color="k", ls="--", lw=0.8)
ax.set_xlabel("x del receptor [m]"); ax.set_ylabel("Dispersión de retardo RMS [ns]")
ax.set_title("Dispersión de retardo desde Tx0 a lo largo de y = 300 m")
ax.legend()
plt.show()

**Preguntas**

1. ¿Por qué la dispersión de retardo cambia bruscamente al cruzar $x = 500$ m?
2. Sin muro solo hay dos rayos (directo y suelo). ¿Por qué aun así la dispersión de retardo es distinta de cero? ¿Es grande o pequeña frente a la de un canal con muro?
3. El **ancho de banda de coherencia** se estima como $B_c \approx 1/(5\,\sigma_\tau)$. Calcúlalo para el valor que obtuviste en el punto $(400, 300)$. ¿Un canal de 20 MHz sería selectivo en frecuencia?

### Tu turno

- **(a)** Elige otro receptor, por ejemplo `rx = (700, 300, 1.5)` (al otro lado del muro respecto a Tx0), y repite el análisis de rayos. ¿Qué rayos existen y por qué?
- **(b)** Cambia `rx_height` (tercer valor de `rx`) a 10 m y observa cómo cambian los retardos del rayo de suelo.
- **(c)** Exporta tu simulación de la Práctica 2 con `res_con.export_csv("cobertura.csv")` y `res_con.export_json("resumen.json")`, y ábrelos con una hoja de cálculo o con `np.genfromtxt`.

In [ ]:
# Espacio para tus ejercicios (a)-(c)

---
## Práctica 4: suma coherente, antenas sectoriales y SINR

### 4.1 Suma incoherente vs. suma coherente

Hasta ahora la potencia recibida era la **suma incoherente** de los rayos, $P = \sum_k |a_k|^2$.
En realidad los rayos son ondas de la misma frecuencia que se suman con su **fase**: $P = \left|\sum_k a_k\right|^2$.
Según la diferencia de caminos, se refuerzan o se cancelan: es el **desvanecimiento rápido** (*fast fading*).

En OpenRPS la suma coherente se activa con `coherent=True`. Empezamos por el caso más simple: una estación base
sin muros, es decir, solo **rayo directo + rayo reflejado en el suelo**. Es exactamente el modelo **Two-Ray** de la Práctica 1.

In [ ]:
tx = Transmitter(tx_id=0, x=0, y=0, z=25, tx_power_dbm=43, frequency_hz=2.4e9, antenna_gain_dbi=12)
sim1 = PropagationSimulator()
sim1.add_transmitter(tx)

d = np.logspace(1, 4, 4000)                                    # receptor de 10 m a 10 km, a 1.5 m de altura
puntos = np.column_stack([d, np.zeros_like(d), np.full_like(d, 1.5)])
rayos = sim1.ray_tracer.trace_points(tx, puntos)               # rayo directo + rayo de suelo en cada punto

# Pérdida de enlace P_t - P_r (incluye la ganancia de la antena), con los dos tipos de suma
pl_incoh = tx.tx_power_dbm - 10 * np.log10(rayos.total_power_linear())
pl_coh = tx.tx_power_dbm - 10 * np.log10(rayos.total_power_linear(coherent=True))
pl_2ray = TwoRayGroundReflection().path_loss(d, tx.frequency_hz, ht_m=tx.z, hr_m=1.5, tx_gain_dbi=tx.antenna_gain_dbi)

lam = 299792458.0 / tx.frequency_hz
d_quiebre = 4 * tx.z * 1.5 / lam                               # punto de quiebre 4·ht·hr/λ

fig, ax = plt.subplots(figsize=(9, 4.8))
ax.semilogx(d, pl_coh, lw=0.9, label="Ray tracing, suma coherente")
ax.semilogx(d, pl_2ray, "k:", lw=1.2, label="Modelo Two-Ray")
ax.semilogx(d, pl_incoh, lw=2, label="Ray tracing, suma incoherente")
ax.axvline(d_quiebre, color="gray", ls="--", lw=0.8)
ax.annotate("punto de quiebre\n4·ht·hr/λ", (d_quiebre, pl_coh.min()), textcoords="offset points", xytext=(6, 0), fontsize=8, va="top")
ax.invert_yaxis()
ax.set_xlabel("Distancia horizontal [m]"); ax.set_ylabel("Pérdida de enlace [dB]")
ax.set_title("Rayo directo + suelo a 2.4 GHz (ht = 25 m, hr = 1.5 m)")
ax.legend()
plt.show()

print(f"Punto de quiebre: {d_quiebre:.0f} m")
print(f"Diferencia máxima entre la suma coherente y Two-Ray: {np.max(np.abs(pl_coh - pl_2ray)):.1e} dB")
print(f"Coherente - incoherente: entre {np.min(pl_incoh - pl_coh):+.1f} dB (nulo) y {np.max(pl_incoh - pl_coh):+.1f} dB (refuerzo)")

Ahora el mapa completo de la Práctica 2 (dos estaciones y el muro), con los dos tipos de suma.
Con `coherent=True` cada píxel tiene tres rayos por estación que interfieren entre sí.

In [ ]:
sim = construir_simulador(con_muro=True)
zona = dict(x_range=(0, 1000), y_range=(0, 1000), resolution_m=5.0)
res_incoh = sim.run_raytracing_simulation(**zona)
res_coh = sim.run_raytracing_simulation(**zona, coherent=True)

ext4 = [0, 1000, 0, 1000]
dif = res_coh.max_rx_power_dbm - res_incoh.max_rx_power_dbm

fig, axs = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)
vmin, vmax = np.nanpercentile(res_coh.max_rx_power_dbm, 1), np.nanmax(res_incoh.max_rx_power_dbm)
for ax, res, titulo in [(axs[0], res_incoh, "suma incoherente"), (axs[1], res_coh, "suma coherente")]:
    im = ax.imshow(res.max_rx_power_dbm, origin="lower", extent=ext4, vmin=vmin, vmax=vmax, cmap="viridis")
    ax.set_title(f"Potencia recibida [dBm]: {titulo}")
fig.colorbar(im, ax=axs[:2], shrink=0.85, label="dBm")
im2 = axs[2].imshow(dif, origin="lower", extent=ext4, cmap="RdBu", vmin=-15, vmax=15)
axs[2].set_title("Coherente - incoherente [dB]")
fig.colorbar(im2, ax=axs[2], shrink=0.85, label="dB")
for ax in axs:
    ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")
plt.show()

print(f"Diferencia coherente - incoherente: mínimo {np.nanmin(dif):.1f} dB, máximo {np.nanmax(dif):+.1f} dB")
print(f"Píxeles con un desvanecimiento de más de 10 dB: {100 * np.mean(dif < -10):.1f} %")
print(f"Media (en mW) de P_coherente / P_incoherente: {np.nanmean(10 ** (dif / 10)):.2f}")

**Preguntas**

1. En la gráfica de la pérdida, ¿por qué la curva coherente oscila alrededor de la incoherente antes del punto de quiebre y deja de hacerlo después? ¿Qué pendiente (dB/década) tiene cada curva a partir de ahí? Compárala con la Práctica 1.
2. La suma coherente supera a la incoherente como mucho en unos 3 dB, pero puede quedar decenas de dB por debajo. ¿Por qué esa asimetría? (Pista: compara $(|a_1| + |a_2|)^2$ y $(|a_1| - |a_2|)^2$ con $|a_1|^2 + |a_2|^2$.)
3. En el mapa coherente se ven anillos anchos lejos de cada estación (los lóbulos del modelo Two-Ray de la gráfica anterior) y, cerca de las estaciones y del muro, un aspecto "granulado". Ahí los máximos y mínimos están separados por distancias mucho menores que los 5 m de la cuadrícula (hasta del orden de la longitud de onda, 12.5 cm a 2.4 GHz). ¿Estamos viendo la forma real del desvanecimiento o una muestra de él? ¿Qué mapa usarías para planificar la cobertura y cuál para estimar el margen de desvanecimiento?
4. La media de $P_\text{coh}/P_\text{incoh}$ sale cercana a 1. ¿Qué te dice eso sobre el significado físico de la suma incoherente?

> **Limitación:** las fases salen de una geometría perfecta (suelo y muro lisos, posiciones exactas). El simulador reproduce bien el *fenómeno* y su estadística, pero no puede predecir en qué punto exacto de una calle real habrá un nulo. La dispersión de retardo y angular no cambian con `coherent=True`: siempre ponderan cada rayo con su potencia $|a_k|^2$.

### 4.2 SINR con antenas isotrópicas y sectoriales

Que llegue mucha potencia no basta: importa cuánta llega **respecto a la interferencia** de las otras estaciones que usan la misma frecuencia, y al **ruido** del receptor:

$$\text{SINR} = \frac{P_\text{servidor}}{\sum_{i \neq \text{servidor}} P_i + N}, \qquad N = k\,T\,B\,F$$

`resultado.compute_sinr(noise_figure_db=7, bandwidth_hz=20e6)` lo calcula en cada píxel (con $T$ = 290 K; $N \approx -94$ dBm para 20 MHz).

Comparamos dos estaciones co-canal con antenas **isotrópicas** y con antenas **sectoriales** (diagrama parabólico de 3GPP, 65° de ancho de haz) que miran las dos hacia el este (+x).

In [ ]:
def simular_red(patron, azimut=(0.0, 0.0), **opciones):
    sim = PropagationSimulator()
    for tx_id, x in [(0, 200), (1, 800)]:
        sim.add_transmitter(Transmitter(tx_id=tx_id, x=x, y=500, z=25, tx_power_dbm=43, frequency_hz=2.4e9,
                                        antenna_gain_dbi=12, pattern=patron, azimuth_deg=azimut[tx_id]))
    return sim.run_raytracing_simulation(x_range=(0, 1000), y_range=(0, 1000), resolution_m=10.0, **opciones)

res_iso = simular_red("isotropic")
res_sec = simular_red("sector")                      # azimut 0° = las dos antenas miran hacia +x

sinr_iso = res_iso.compute_sinr(noise_figure_db=7, bandwidth_hz=20e6)
sinr_sec = res_sec.compute_sinr(noise_figure_db=7, bandwidth_hz=20e6)

print(f"Potencia de ruido: {res_iso.sinr_settings['noise_power_dbm']:.1f} dBm")
print(f"{'':12s} {'SINR mediano':>13s} {'percentil 5':>12s} {'área con SINR > 10 dB':>22s}")
for nombre, sinr in [("Isotrópica", sinr_iso), ("Sectorial", sinr_sec)]:
    print(f"{nombre:12s} {np.nanmedian(sinr):10.1f} dB {np.nanpercentile(sinr, 5):9.1f} dB {100 * np.mean(sinr > 10):19.1f} %")

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)
vmin, vmax = -5, 40
for ax, sinr, titulo in [(axs[0], sinr_iso, "antenas isotrópicas"), (axs[1], sinr_sec, "antenas sectoriales (→ este)")]:
    im = ax.imshow(sinr, origin="lower", extent=ext4, vmin=vmin, vmax=vmax, cmap="viridis")
    ax.contour(res_iso.x_coords, res_iso.y_coords, sinr, levels=[0, 10], colors="w", linewidths=0.8)
    ax.plot([200, 800], [500, 500], "r^", ms=9)
    ax.set_title(f"SINR [dB]: {titulo}")
    ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")
fig.colorbar(im, ax=axs[:2], shrink=0.85, label="dB (contornos: 0 y 10 dB)")

for sinr, nombre in [(sinr_iso, "Isotrópica"), (sinr_sec, "Sectorial")]:
    valores = np.sort(sinr[np.isfinite(sinr)])
    axs[2].plot(valores, np.arange(1, valores.size + 1) / valores.size, label=nombre)
axs[2].set_xlabel("SINR [dB]"); axs[2].set_ylabel("Fracción del área con SINR menor")
axs[2].set_title("Distribución acumulada (CDF) del SINR")
axs[2].legend()
plt.show()

**Preguntas**

1. Con antenas isotrópicas, ¿dónde está la línea de SINR = 0 dB y por qué? ¿Dónde está con las antenas sectoriales?
2. ¿En qué zonas mejora el SINR al usar sectores y en cuáles empeora? Explícalo con el diagrama: la antena atenúa hasta 30 dB hacia atrás, tanto la señal útil (detrás de su propia estación) como la interferencia (hacia la celda vecina).
3. Compara `res_sec.interference_plus_noise_dbm` con la potencia de ruido. ¿El sistema está limitado por ruido o por interferencia? ¿Serviría de algo subir la potencia de las dos estaciones 10 dB?
4. Detrás de Tx0 (x < 200 m) el SINR con sectores es parecido al de las antenas isotrópicas, aunque la antena apunta hacia el otro lado. ¿Por qué? Compara `res_sec.max_rx_power_dbm` y `res_iso.max_rx_power_dbm` en esa zona: ¿es igual de buena la cobertura? ¿Qué pasaría allí con un receptor más ruidoso o un ancho de banda mucho mayor?

### Tu turno

- **(a)** Repite el mapa coherente de 4.1 en una zona pequeña con mucha más resolución, p. ej. `x_range=(390, 400), y_range=(295, 305), resolution_m=0.02`. ¿Qué forma tienen ahora los desvanecimientos? ¿Cuánto mide la distancia entre dos mínimos?
- **(b)** Gira Tx1 para que mire a Tx0: `simular_red("sector", azimut=(0, 180))`. ¿Cómo cambia el mapa de SINR? ¿Y con `azimut=(180, 0)` (espalda con espalda)?
- **(c)** Cambia la frecuencia de Tx1 a 2.6 GHz (otra portadora). ¿Qué pasa con el SINR? ¿Qué coste tiene esa solución?
- **(d)** Añade desvanecimiento por sombra: `simular_red("sector", shadow_fading_sigma_db=8, seed=1)` y dibuja de nuevo la CDF. ¿Cómo cambia el percentil 5? Prueba otra semilla: ¿cambia el mapa?, ¿y la CDF?
- **(e)** Calcula el SINR con `bandwidth_hz=200e3` (un canal estrecho) y con `noise_figure_db=3`. ¿En qué zonas cambia el resultado?

In [ ]:
# Espacio para tus ejercicios (a)-(e)

---
## Resumen

| Concepto | Dónde lo viste |
|---|---|
| Pérdida de trayecto y exponente $n$ | Práctica 1 |
| Rango de validez de los modelos empíricos | Práctica 1 (Hata y COST-231) |
| Cobertura, celda servidora, efecto del entorno | Práctica 2 |
| CIR, retardo medio, dispersión de retardo y angular | Prácticas 2 y 3 |
| Suma incoherente vs. coherente, desvanecimiento rápido, modelo Two-Ray | Práctica 4 |
| Antenas sectoriales, interferencia co-canal, ruido y SINR | Práctica 4 |
| Limitaciones: muros que no bloquean, un solo rebote, terreno plano, fases idealizadas | Notas en las prácticas y en el README |

Revisa la sección *Limitaciones físicas* del README del proyecto antes de usar estos resultados fuera del aula.